In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

yt = pd.read_csv('../data/cleaned/youtube_clean.csv')
ig = pd.read_csv('../data/cleaned/instagram_clean.csv')
print('Data loaded successfully.')

In [ ]:
def normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - mn) / (mx - mn)

yt['score'] = (
    0.4 * normalize(yt['engagement_rate']) +
    0.3 * normalize(yt['views']) +
    0.2 * normalize(yt['likes']) +
    0.1 * normalize(yt['comments'])
).round(4)

top_content = yt.nlargest(10, 'score')[['title','views','likes','comments','engagement_rate','score']]
top_content['title'] = top_content['title'].str[:40]
print('Top 10 videos by combined score:')
print(top_content.to_string(index=False))

In [ ]:
yt['published_at'] = pd.to_datetime(yt['published_at'])
yt['year_month'] = yt['published_at'].dt.to_period('M').astype(str)

monthly = yt.groupby('year_month').agg(
    total_views=('views','sum'),
    avg_engagement=('engagement_rate','mean'),
    post_count=('video_id','count')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(monthly['year_month'], monthly['total_views'], marker='o', color='steelblue')
axes[0].set_title('Monthly total views')
axes[0].set_xlabel('Month')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['year_month'], monthly['avg_engagement'], marker='s', color='darkorange')
axes[1].set_title('Monthly avg engagement rate')
axes[1].set_xlabel('Month')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/monthly_trend.png', dpi=150)
plt.show()

In [ ]:
corr_cols = ['views','likes','comments','engagement_rate','hour']
corr = yt[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation matrix (YouTube metrics)')
plt.tight_layout()
plt.savefig('../data/processed/correlation.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    ig['reach'], ig['engagement_rate'],
    c=ig['like_count'], cmap='YlOrRd', alpha=0.7, s=70, edgecolors='white'
)
plt.colorbar(scatter, label='Like count')
ax.set_xlabel('Reach')
ax.set_ylabel('Engagement rate')
ax.set_title('Instagram: Reach vs Engagement rate (color = likes)')
plt.tight_layout()
plt.savefig('../data/processed/ig_reach_engagement.png', dpi=150)
plt.show()

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

yt_best_day  = yt.groupby('day_of_week')['engagement_rate'].mean().reindex(day_order).idxmax()
yt_best_hour = yt.groupby('hour')['engagement_rate'].mean().idxmax()
ig_best_day  = ig.groupby('day_of_week')['engagement_rate'].mean().reindex(day_order).idxmax()
ig_best_hour = ig.groupby('hour')['engagement_rate'].mean().idxmax()

top_yt = yt.nlargest(1, 'score').iloc[0]

print('='*50)
print('       CONTENT STRATEGY REPORT')
print('='*50)
print()
print('[YouTube]')
print(f'  Best day to post     : {yt_best_day}')
print(f'  Best hour to post    : {yt_best_hour}:00')
print(f'  Avg engagement rate  : {yt["engagement_rate"].mean():.2%}')
print(f'  Best performing video: {top_yt["title"][:50]}')
print(f'  Its score            : {top_yt["score"]:.3f}')
print()
print('[Instagram]')
print(f'  Best day to post     : {ig_best_day}')
print(f'  Best hour to post    : {ig_best_hour}:00')
print(f'  Avg engagement rate  : {ig["engagement_rate"].mean():.2%}')
print(f'  Best media type      : {ig.groupby("media_type")["engagement_rate"].mean().idxmax()}')
print()
print('Charts saved to data/processed/')